# Import libraries

In [1]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from pprint import pprint
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

True

# Get original files

In this exercise, I'll use only the chapter 2 of the book

In [2]:
# Load all PDFs from data folder
data_dir = "data"
loader = PyPDFLoader(os.path.join(data_dir, "book_chapter_02.pdf"))
docs = loader.load()

print(f"Quantity of docs: {len(docs)}")
print("\nExample of doc:")
docs[0]

Quantity of docs: 34

Example of doc:


Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110007', 'source': 'data\\book_chapter_02.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}, page_content='4 CHAPTER 2 • W ORDS AND TOKENS\nCHAPTER\n2\nWords and Tokens\nUser: I need some help, that much seems certain.\nELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP\nUser: Perhaps I could learn to get along with my mother.\nELIZA: TELL ME MORE ABOUT YOUR FAMILY\nUser: My mother takes care of me.\nELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU\nUser: My father.\nELIZA: YOUR FATHER\nUser: You are like my father in some ways.\nWeizenbaum (1966)\nThe dialogue above is fromELIZA, an early natural language processing systemELIZA\nthat could carry on a limited conversation with a user by imitating the responses of\na Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple\nprogram that uses pattern matching on words to recognize phrases like “I need X”\nand change the word

## Save only useful metadata 

In [3]:
keys_to_keep =["creationdate", "source", "total_pages", "page"]

for doc in docs:
    # Filter the dictionary
    doc.metadata = {k: v for k, v in doc.metadata.items() if k in keys_to_keep}

# Initialize LLM

In [4]:
api_key = os.getenv("API_KEY")
llm = model = ChatGoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=1.0,
    max_tokens=1500,
    timeout=None,
    max_retries=2
)

In the next cell, I test the llm:

In [5]:
llm.invoke("hello")

AIMessage(content='Hello there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5d86-9dce-76d1-88e3-f624f10d5bc3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 10, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}})

# Create tags to apply to the chunks

In [6]:
class Tags(BaseModel):
    volume: str = Field(
        description="The major part of the book the chunk belongs to. The possible values must be: 'I: Large Language Models', 'II: Annotating Linguistic Structure'",
        default=None
    )
    chapter_number: int = Field(
        description="The numerical index of the chapter.",
        default=None
    ),
    section_title: str = Field(
        description="The specific section or subsection header the chunk falls under.",
        default="General"
    )
    content_type: str = Field(
        description="The functional nature of the text fragment. The possible values must be: 'Narrative', 'Summary', 'Historical Notes', 'Examples', 'Exercises', 'Table of Contents', 'Algorithm Description'",
        default="Narrative"
    )
    contains_math_latex: bool = Field(
        default=False,
        description="Whether the chunk contains formal mathematical notation, probability equations, or LaTeX-style formulas.",
    )
    contains_regex: bool = Field(
        default=False,
        description="Whether the chunk includes regular expression patterns or explains regex syntax.",
    )
    contains_code_or_cli: bool = Field(
        default=False,
        description="Whether the text includes Python code, Unix commands (tr, sort, uniq), or pseudocode for algorithms.",
    )
    contains_table: bool = Field(
        default=False,
        description="Whether the fragment includes a data table, such as word counts, probabilities, or ASCII/Unicode mappings.",
    )
    talks_about_tokenization: bool = Field(
        default=False,
        description="Whether the chunk discusses the process of segmenting running input text into tokens[cite: 377, 1268].",
    )
    talks_about_bpe: bool = Field(
        default=False,
        description="Whether the chunk discusses the data-driven induction of subword tokens using Byte-Pair Encoding[cite: 398, 1274].",
    )
    talks_about_unicode: bool = Field(
        default=False,
        description="Whether the chunk discusses character representation systems like Unicode code points or UTF-8 encoding[cite: 274, 300, 1272, 1273].",
    )
    talks_about_edit_distance: bool = Field(
        default=False,
        description="Whether the chunk discusses measuring string similarity via minimum operations like insertion, deletion, and substitution[cite: 1020, 1279].",
    )
    talks_about_ngrams: bool = Field(
        default=False,
        description="Whether the chunk discusses sequences of $n$ words or Markov-based models that look $n-1$ words into the past[cite: 1355, 1407, 1967].",
    )
    talks_about_language_modeling: bool = Field(
        default=False,
        description="Whether the chunk discusses the general machine learning task of predicting upcoming words or assigning probabilities to sequences[cite: 1335, 1965].",
    )
    talks_about_evaluation_perplexity: bool = Field(
        default=False,
        description="Whether the chunk discusses intrinsic evaluation using the inverse probability of a test set normalized by length[cite: 1576, 1577, 1970].",
    )
    talks_about_smoothing_interpolation: bool = Field(
        default=False,
        description="Whether the chunk discusses handling unseen events by shifting probability mass or combining different order n-grams[cite: 1754, 1827, 1972, 1973].",
    )
    talks_about_morphology: bool = Field(
        default=False,
        description="Whether the chunk discusses morphemes as the minimal meaning-bearing units of language[cite: 191, 192, 1271].",
    )
    linguistic_focus: str = Field(
        description="Specific languages or language families discussed as examples (e.g., Chinese, English, Spanish, Vietnamese).",
        default="General"
    )
    importance_score: str = Field(
        description="The pedagogical value of the chunk for understanding core NLP foundations. The possible values must be: 'High', 'Medium', 'Low'",
        default="Medium"
    )


pprint(Tags.schema())

{'properties': {'chapter_number': {'title': 'Chapter Number',
                                   'type': 'integer'},
                'contains_code_or_cli': {'default': False,
                                         'description': 'Whether the text '
                                                        'includes Python code, '
                                                        'Unix commands (tr, '
                                                        'sort, uniq), or '
                                                        'pseudocode for '
                                                        'algorithms.',
                                         'title': 'Contains Code Or Cli',
                                         'type': 'boolean'},
                'contains_math_latex': {'default': False,
                                        'description': 'Whether the chunk '
                                                       'contains formal '
                          

C:\Users\alejo\AppData\Local\Temp\ipykernel_13348\2494597465.py:80: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  pprint(Tags.schema())
c:\Users\alejo\code\ai-exercises\.venv\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=False, default=None, description='The numerical index of the chapter.'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Create the tagger to apply to the chunks:

In [7]:
tagger = llm.with_structured_output(Tags)

Example with a string:

In [8]:
result = tagger.invoke("the language model is very important to build strong systemns")
pprint(result)

Tags(volume='I: Large Language Models', chapter_number=(FieldInfo(annotation=NoneType, required=False, default=None, description='The numerical index of the chapter.'),), section_title='General', content_type='Narrative', contains_math_latex=False, contains_regex=False, contains_code_or_cli=False, contains_table=False, talks_about_tokenization=False, talks_about_bpe=False, talks_about_unicode=False, talks_about_edit_distance=False, talks_about_ngrams=False, talks_about_language_modeling=False, talks_about_evaluation_perplexity=False, talks_about_smoothing_interpolation=False, talks_about_morphology=False, linguistic_focus='General', importance_score='High')


Examples with docs:

In [14]:
inputs = [doc.page_content for doc in docs[0:(len(docs)//6)]]
results = tagger.batch(inputs=inputs, return_exceptions=True)
results

[Tags(volume='II: Annotating Linguistic Structure', chapter_number=2, section_title='Words and Tokens', content_type='Narrative', contains_math_latex=False, contains_regex=True, contains_code_or_cli=False, contains_table=False, talks_about_tokenization=True, talks_about_bpe=True, talks_about_unicode=True, talks_about_edit_distance=True, talks_about_ngrams=False, talks_about_language_modeling=False, talks_about_evaluation_perplexity=False, talks_about_smoothing_interpolation=False, talks_about_morphology=True, linguistic_focus='Vietnamese, Cantonese, Turkish', importance_score='High'),
 Tags(volume=None, chapter_number=2, section_title='Words', content_type='Narrative', contains_math_latex=False, contains_regex=False, contains_code_or_cli=False, contains_table=False, talks_about_tokenization=True, talks_about_bpe=False, talks_about_unicode=False, talks_about_edit_distance=False, talks_about_ngrams=False, talks_about_language_modeling=False, talks_about_evaluation_perplexity=False, talks

In [50]:
docs[2].page_content.split("\n")

['6 CHAPTER 2 • W ORDS AND TOKENS',
 'Corpus Types = |V | Instances = N',
 'Shakespeare 31 thousand 884 thousand',
 'Brown corpus 38 thousand 1 million',
 'Switchboard telephone conversations 20 thousand 2.4 million',
 'COCA 2 million 440 million',
 'Google n-grams 13 million 1 trillion',
 'Figure 2.1 Rough numbers of wordform types and instances for some English language',
 'corpora. The largest, the Google n-grams corpus, contains 13 million types, but this count',
 'only includes types appearing 40 or more times, so the true number would be much larger.',
 'The distinctions get even harder to make once we start to think about other lan-',
 'guages. For example the writing systems of languages like Chinese, Japanese, and',
 'Thai simply don’t have orthographic words at all! That is, they don’t use spaces to',
 'mark potential word-boundaries. In Chinese, for example, words are composed of',
 'characters (called hanzi in Chinese). Each character generally represents a singlehanzi',
 '

In [48]:
for key in list(results[2].model_fields.keys()):
    print(f"{key}: {getattr(results[2], key)}")


volume: II: Annotating Linguistic Structure
chapter_number: 2
section_title: WORDS AND TOKENS
content_type: Narrative
contains_math_latex: False
contains_regex: False
contains_code_or_cli: False
contains_table: True
talks_about_tokenization: True
talks_about_bpe: False
talks_about_unicode: False
talks_about_edit_distance: False
talks_about_ngrams: False
talks_about_language_modeling: False
talks_about_evaluation_perplexity: False
talks_about_smoothing_interpolation: False
talks_about_morphology: False
linguistic_focus: Chinese
importance_score: High


C:\Users\alejo\AppData\Local\Temp\ipykernel_13348\316331021.py:1: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  for key in list(results[2].model_fields.keys()):


In [ ]:
# To do:
# Indexar los documentos
# docs_with_tags = [
#     Document(
#         page_content=doc.page_content,
#         metadata={
#             **doc.metadata,
#             **result.get("text").dict(),
#         },
#     )
#     for doc, result in zip(docs, tagging_results)
#     if not isinstance(result, Exception)
# ]

# f"Documents with tags: {len(docs_with_tags)}"